In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are imdependent and often  enjow their own space.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="GoldFish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source":"fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source":"mammal-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='GoldFish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000205B36FEDA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000205B36FF2E0>, model_name='Llama3-8b-8192')

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1962.47it/s]


In [6]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore

In [7]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='84daa7a2-5cea-4e60-be84-df65a27f1972', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
  0.8950580358505249),
 (Document(id='2e33dada-655d-4a4a-a385-bfd1264c7c7d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='9c00ce6f-66d2-4a45-9cd7-066d9b0a2fb2', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.595690369606018),
 (Document(id='507bd8ae-bf0a-48f9-bb7c-f8cb465de8a3', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.665792465209961)]

In [8]:
await vectorstore.asimilarity_search("cat")

[Document(id='84daa7a2-5cea-4e60-be84-df65a27f1972', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
 Document(id='2e33dada-655d-4a4a-a385-bfd1264c7c7d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='9c00ce6f-66d2-4a45-9cd7-066d9b0a2fb2', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='507bd8ae-bf0a-48f9-bb7c-f8cb465de8a3', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [9]:
### Retrivers
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriver=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriver.batch(["cat","dog","parrot"])

[[Document(id='84daa7a2-5cea-4e60-be84-df65a27f1972', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.')],
 [Document(id='2e33dada-655d-4a4a-a385-bfd1264c7c7d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='507bd8ae-bf0a-48f9-bb7c-f8cb465de8a3', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]]

In [10]:
## prefer this approach
retriver=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriver.batch(["cat","dog","parrot"])


[[Document(id='84daa7a2-5cea-4e60-be84-df65a27f1972', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.')],
 [Document(id='2e33dada-655d-4a4a-a385-bfd1264c7c7d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='507bd8ae-bf0a-48f9-bb7c-f8cb465de8a3', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]]

In [14]:
### integrating retriver and chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using provided context only.
{question}

Context:
{context}
"""
prompt=ChatPromptTemplate.from_messages([("human",message)])

rag_chain={"context":retriver, "question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("Tell me about cats")
print(response.content)

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}